In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd, torch
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:7.0f}с] {m}", flush=True)
name = torch.cuda.get_device_name(0); log(f"GPU: {name}")
if "T4" not in name and "L4" not in name and "A100" not in name:
    raise SystemExit(f"нужна T4, выдали {name}")
code = os.path.dirname(glob.glob("/kaggle/input/**/cross_encoder.py", recursive=True)[0])
os.makedirs("/kaggle/working/src", exist_ok=True)
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.cross_encoder import build_product_texts
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer
import torch.nn.functional as F

# Те же 400 тысяч пар, что и в прошлых проходах: то же вычитание обученных пар и то же зерно.
pack = os.path.dirname(glob.glob("/kaggle/input/**/llm_train.parquet", recursive=True)[0])
big = os.path.dirname(glob.glob("/kaggle/input/**/llm_pairs_2m.parquet", recursive=True)[0])
seen = pd.read_parquet(pack + "/llm_train.parquet", columns=["id1", "id2"])
pairs = pd.read_parquet(big + "/llm_pairs_2m.parquet")
items = pd.read_parquet(big + "/llm_items_2m.parquet")
key = lambda d: pd.MultiIndex.from_arrays([d.id1.to_numpy(), d.id2.to_numpy()])
clean = pairs[~key(pairs).isin(key(seen))].reset_index(drop=True)
if len(clean) > 400_000:
    clean = clean.sample(400_000, random_state=2026).reset_index(drop=True)
used = pd.unique(np.concatenate([clean.id1.to_numpy(), clean.id2.to_numpy()]))
items = items[items.id.isin(used)].reset_index(drop=True)
log(f"пар {len(clean):,}, карточек {len(items):,}")
texts = build_product_texts(items, "compact")
left = clean.id1.map(texts).fillna("").astype(str).to_numpy()
right = clean.id2.map(texts).fillna("").astype(str).to_numpy()
order = np.argsort(np.fromiter((len(a)+len(b) for a, b in zip(left, right)), dtype=np.int32, count=len(left)))
log("тексты готовы")

def unpack_flat(prefix, where):
    root = os.path.dirname(glob.glob(f"/kaggle/input/**/{prefix}__model.safetensors", recursive=True)[0])
    os.makedirs(where, exist_ok=True)
    for f in ("model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json"):
        dst = f"{where}/{f}"
        if not os.path.exists(dst) and os.path.exists(f"{root}/{prefix}__{f}"):
            os.symlink(f"{root}/{prefix}__{f}", dst)
    return where

def find_dir(pattern):
    hits = glob.glob(pattern, recursive=True)
    return os.path.dirname(hits[0]) if hits else None

def score_cross(path, batch=256):
    tok = AutoTokenizer.from_pretrained(path, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        path, local_files_only=True, dtype=torch.float16).cuda().eval()
    out = np.empty(len(left), dtype=np.float32)
    with torch.inference_mode():
        for start in range(0, len(order), batch):
            rows = order[start:start+batch]
            enc = tok(left[rows].tolist(), right[rows].tolist(), padding=True, truncation=True,
                      max_length=256, pad_to_multiple_of=8, return_tensors="pt").to("cuda")
            out[rows] = model(**enc).logits.squeeze(-1).float().cpu().numpy()
    del model; torch.cuda.empty_cache(); gc.collect()
    return out

def score_bi(path, batch=512):
    """Двухбашенная: товар кодируется один раз, скор пары — косинус."""
    tok = AutoTokenizer.from_pretrained(path, local_files_only=True)
    model = AutoModel.from_pretrained(path, local_files_only=True, dtype=torch.float16).cuda().eval()
    ids = items.id.tolist(); strings = [texts.get(int(i), "") for i in ids]
    vecs = np.empty((len(ids), model.config.hidden_size), dtype=np.float32)
    with torch.inference_mode():
        for start in range(0, len(strings), batch):
            chunk = strings[start:start+batch]
            enc = tok(chunk, padding=True, truncation=True, max_length=128,
                      pad_to_multiple_of=8, return_tensors="pt").to("cuda")
            hidden = model(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).half()
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-6)
            vecs[start:start+len(chunk)] = F.normalize(pooled.float(), dim=-1).cpu().numpy()
    del model; torch.cuda.empty_cache(); gc.collect()
    position = {int(v): row for row, v in enumerate(ids)}
    a = np.array([position.get(int(i), -1) for i in clean.id1])
    b = np.array([position.get(int(i), -1) for i in clean.id2])
    good = (a >= 0) & (b >= 0)
    out = np.zeros(len(clean), dtype=np.float32)
    out[good] = (vecs[a[good]] * vecs[b[good]]).sum(-1)
    log(f"  двухбашенная: закодировано {len(ids):,} карточек, пар с обеими сторонами {good.mean():.1%}")
    return out

SOURCES = {
    "ce_spec":   find_dir("/kaggle/input/**/ce_spec/model.safetensors") or unpack_flat("ce_spec", "/kaggle/working/enc/ce_spec"),
    "ce_e5":     find_dir("/kaggle/input/**/ce_e5/model.safetensors") or unpack_flat("ce_e516", "/kaggle/working/enc/ce_e5"),
    "ce_hardneg": find_dir("/kaggle/input/**/ce_hardneg/model.safetensors"),
}
BI_DIR = find_dir("/kaggle/input/**/bi_encoder/model.safetensors")
log(f"каталоги: {SOURCES} | двухбашенная: {BI_DIR}")
for tag, path in SOURCES.items():
    if path is None: raise SystemExit(f"не найден каталог для {tag}")
    t = time.perf_counter()
    np.save(f"/kaggle/working/{tag}.npy", score_cross(path))
    log(f"  {tag}: {time.perf_counter()-t:.0f}с")
t = time.perf_counter()
np.save("/kaggle/working/ce_bi.npy", score_bi(BI_DIR))
log(f"  ce_bi: {time.perf_counter()-t:.0f}с")
clean.to_parquet("/kaggle/working/clean_pairs_v4.parquet", index=False)
items.to_parquet("/kaggle/working/clean_items_v4.parquet", index=False)
log("готово")
